In [1]:
import sys
print(f"Python: {sys.version}")
print(f"Running inside: {sys.prefix}")
print("Course environment is ready!")

Python: 3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:19:04) [GCC 13.3.0]
Running inside: /opt/conda
Course environment is ready!


In [3]:
%pwd

'/home/jovyan/notebooks'

In [2]:
%%file producer.py
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

stores = ["Warsaw", "Krakow", "Gdansk", "Wroclaw"]
categories = ["electronics", "clothing", "food", "books"]

def generate_transaction(tx_number):
    return {
        "tx_id": f"TX{tx_number:04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": datetime.now().isoformat()
    }

for i in range(1, 51):
    transaction = generate_transaction(i)
    producer.send("transactions", value=transaction)
    print(f"Sent: {transaction}")
    time.sleep(1)

producer.flush()
print("Done sending 50 transactions.")

Overwriting producer.py


In [4]:
%%file /home/jovyan/notebooks/lab1_kafka/consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for message in consumer:
    tx = message.value
    if tx["amount"] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Writing /home/jovyan/notebooks/lab1_kafka/consumer_filter.py


In [5]:
%%file /home/jovyan/notebooks/lab1_kafka/consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening and enriching transactions...")

for message in consumer:
    tx = message.value
    amount = tx["amount"]

    if amount > 3000:
        tx["risk_level"] = "HIGH"
    elif amount > 1000:
        tx["risk_level"] = "MEDIUM"
    else:
        tx["risk_level"] = "LOW"

    print(tx)

Writing /home/jovyan/notebooks/lab1_kafka/consumer_enrich.py


In [6]:
%%file /home/jovyan/notebooks/lab1_kafka/consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

print("Counting transactions per store...")

for message in consumer:
    tx = message.value
    store = tx["store"]
    amount = tx["amount"]

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- STORE SUMMARY ---")
        print("Store | Count | Total Amount | Avg Amount")
        for s in sorted(store_counts):
            avg_amount = total_amount[s] / store_counts[s]
            print(f"{s} | {store_counts[s]} | {total_amount[s]:.2f} | {avg_amount:.2f}")

Writing /home/jovyan/notebooks/lab1_kafka/consumer_count.py


In [7]:
%%file /home/jovyan/notebooks/lab1_kafka/consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "total": 0.0,
    "min": float("inf"),
    "max": float("-inf")
})

msg_count = 0

print("Tracking category statistics...")

for message in consumer:
    tx = message.value
    category = tx["category"]
    amount = tx["amount"]

    stats[category]["count"] += 1
    stats[category]["total"] += amount
    stats[category]["min"] = min(stats[category]["min"], amount)
    stats[category]["max"] = max(stats[category]["max"], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- CATEGORY STATS ---")
        print("Category | Count | Total Revenue | Min Amount | Max Amount")
        for c in sorted(stats):
            s = stats[c]
            print(f"{c} | {s['count']} | {s['total']:.2f} | {s['min']:.2f} | {s['max']:.2f}")

Writing /home/jovyan/notebooks/lab1_kafka/consumer_stats.py


In [8]:
# 1. What happens if you start consumer_filter.py AFTER the producer has finished?
# Because auto_offset_reset='earliest', the consumer reads the existing messages
# from the beginning of the topic, so it can still process old transactions.

# 2. What happens if two consumers have the SAME group_id?
# They belong to the same consumer group, so Kafka splits the messages between them.
# Each consumer does not get all messages independently.

# 3. What is the difference between stateless and stateful processing?
# Stateless processing handles each event independently without memory of previous events.
# Example: consumer_filter.py or consumer_enrich.py
# Stateful processing keeps information across events, such as counters or aggregates.
# Example: consumer_count.py or consumer_stats.py

In [9]:
# Homeworks:

In [13]:
%%file /home/jovyan/notebooks/lab1_kafka/consumer_velocity.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_events = defaultdict(list)

print("Listening for velocity anomalies...")

for message in consumer:
    tx = message.value
    user_id = tx["user_id"]
    ts = datetime.fromisoformat(tx["timestamp"])

    user_events[user_id].append(ts)

    cutoff = ts - timedelta(seconds=60)
    user_events[user_id] = [t for t in user_events[user_id] if t >= cutoff]

    if len(user_events[user_id]) > 3:
        print(
            f"ALERT: {user_id} made {len(user_events[user_id])} transactions "
            f"within 60 seconds. Last tx: {tx['tx_id']}"
        )

Overwriting /home/jovyan/notebooks/lab1_kafka/consumer_velocity.py


In [14]:
# 1. What happens if you start consumer_filter.py AFTER the producer has finished?
# Because auto_offset_reset='earliest', the consumer can still read the existing messages
# from the beginning of the topic.

# 2. What happens if two consumers have the SAME group_id?
# They join the same consumer group and Kafka splits the messages between them.
# They do not process the full stream independently.

# 3. What is the difference between stateless and stateful processing?
# Stateless processing handles each event independently without storing past information.
# Example: consumer_filter.py or consumer_enrich.py
# Stateful processing keeps memory across events, such as counters or running statistics.
# Example: consumer_count.py, consumer_stats.py, or consumer_velocity.py